# Домашнє завдання 7. Time-series features і JSONB у PostgreSQL

Mykyta Oliinyk

Дві частини: ML-ознаки часових рядів на денній агрегації NYC Taxi (варіант A, офіційні Parquet-файли TLC) та event-log у JSONB з індексами GIN і expression B-tree, планами `EXPLAIN (ANALYZE, BUFFERS)` і порівнянням із нормалізованою таблицею.

## Завдання 1. Setup і завантаження даних

In [1]:
from IPython.utils.capture import capture_output

with capture_output():
    %pip install -q pgserver psycopg2-binary "psycopg[binary]" sqlalchemy pandas pyarrow requests

import os
import re
import statistics
import tempfile
import time
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
# timezone=UTC: event_ts рахується від TIMESTAMPTZ, тому результат не має залежати від часового поясу машини
engine = create_engine(pg.get_uri(), future=True, connect_args={'options': '-c timezone=UTC'})

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
pd.set_option('display.max_columns', None, 'display.width', 200, 'display.max_colwidth', 130)


def show(sql):
    """Виконує SELECT і повертає DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


def run(sql):
    """Виконує кілька SQL-команд, розділених ';', в одній транзакції."""
    with engine.begin() as conn:
        for statement in re.split(r';\s*(?:\n|$)', sql):
            if statement.strip():
                conn.execute(text(statement))


def run_autocommit(sql):
    """Для команд поза транзакцією, наприклад VACUUM."""
    with engine.connect().execution_options(isolation_level='AUTOCOMMIT') as conn:
        conn.execute(text(sql))


PLANS = {}


def explain(label, sql, runs=7):
    """EXPLAIN (ANALYZE, BUFFERS) кілька разів поспіль. Показує план запуску з медіанним Execution Time,
    бо один замір на мілісекундному масштабі шумний, а перший запуск ще прогріває кеш."""
    statement = text('EXPLAIN (ANALYZE, BUFFERS)\n' + sql)
    with engine.connect() as conn:
        plans = [[row[0] for row in conn.execute(statement)] for _ in range(runs)]
    times = [float(re.search(r'Execution Time: ([\d.]+)', plan[-1]).group(1)) for plan in plans]
    median_run = times.index(statistics.median_low(times))
    plan_text = '\n'.join(plans[median_run])
    PLANS[label] = plan_text
    print(f'[{label}]')
    print(plan_text)
    print(f'Median of {runs} runs shown; Execution Time range {min(times):.3f}..{max(times):.3f} ms')

### Варіант A. Офіційні Parquet-файли NYC TLC

Масштаб завантаження лишається як в умові: близько 500 000 рядків через `to_sql`, далі той самий SQL. Змінено лише спосіб формування зрізу. Файл приблизно впорядкований за часом, тому `head(500_000)` закінчується вже 6 березня (це видно у виводі нижче). На такому зрізі `LAG(..., 7)` і `LAG(..., 30)` завжди NULL, а view з фільтром `trips_lag_30d IS NOT NULL` порожній. Повний березень теж не вирішує проблему: за 31 день `trips_lag_30d` існує лише для останнього дня.

Тому беру лютий і березень 2024: 60 днів до 2024-03-31, той самий горизонт, що у stub-схемі з умови. Замість `head()` з кожного файлу береться випадкова вибірка з однаковою часткою 7.6% (`random_state=42`). Частка спільна для обох місяців, щоб рівень `trip_count` у лютому й березні був порівнянний: `LAG(..., 30)` зіставляє саме ці два місяці.

Наслідок вибірки: `trip_count` у цій роботі означає кількість поїздок у вибірці, приблизно 7.6% реального обсягу, а не повний підрахунок.

Технічні зміни: до `%pip` додано `psycopg[binary]`, бо SQLAlchemy 2.1 для адреси `postgresql://` обирає psycopg 3. Файли читаються по одному, і в пам'яті лишається тільки вибірка. Вони лежать у `tempfile.gettempdir()` замість `/content`, щоб код працював і в Colab, і локально. Додано фільтр за датою посадки: у файлах є 17 рядків поза періодом, зокрема поїздки з 2002 і 2008 років.

In [3]:
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/'
MONTHS = ['2024-02', '2024-03']
PERIOD_START, PERIOD_END = pd.Timestamp('2024-02-01'), pd.Timestamp('2024-04-01')
SAMPLE_FRACTION = 0.076  # однакова частка для обох файлів, разом близько 500 000 рядків

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

data_dir = os.path.join(tempfile.gettempdir(), 'hw7_nyc_tlc')
os.makedirs(data_dir, exist_ok=True)

frames = []
n_source = n_outside = 0
for month in MONTHS:
    file_name = f'yellow_tripdata_{month}.parquet'
    parquet_path = os.path.join(data_dir, file_name)
    if not os.path.exists(parquet_path):
        urlretrieve(BASE_URL + file_name, parquet_path)
    month_df = pd.read_parquet(parquet_path, columns=columns)
    n_source += len(month_df)
    pickup = month_df['tpep_pickup_datetime']
    n_outside += (~pickup.between(PERIOD_START, PERIOD_END, inclusive='left')).sum()
    # Покриття за кодом з умови: де закінчуються перші 500 000 рядків файлу
    print(f'head(500_000) of {month} ends at:', pickup.head(500_000).max())
    frames.append(month_df.sample(frac=SAMPLE_FRACTION, random_state=42))
    del month_df, pickup  # у пам'яті лишається тільки вибірка

df = pd.concat(frames, ignore_index=True)
del frames
n_sampled = len(df)

# Basic cleaning before loading to PostgreSQL
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]
n_valid = len(df)

df = df[df['tpep_pickup_datetime'].between(PERIOD_START, PERIOD_END, inclusive='left')]

print('source rows:              ', n_source)
print('pickup outside period:    ', n_outside)
print('sampled rows:             ', n_sampled)
print('after fare/distance rules:', n_valid)
print('final rows:               ', len(df))

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)
del df

show('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw')

head(500_000) of 2024-02 ends at: 2024-02-06 10:01:56
head(500_000) of 2024-03 ends at: 2024-03-06 05:59:51
source rows:               6590154
pickup outside period:     17
sampled rows:              500852
after fare/distance rules: 481863
final rows:                481862


,n_rows
0,481862


### Daily aggregation `t7_hw_taxi_daily`

SQL з умови. Перед `DROP TABLE` додано `DROP VIEW IF EXISTS`: view з Завдання 2 залежить від таблиці, і без цього другий запуск notebook падає на `DROP TABLE`.

In [4]:
run('''
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;
CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;
ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
''')

show('''
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;
''')

,pickup_zone_id,n_days
0,1,10
1,2,1
2,3,11
3,4,60
4,6,2
5,7,60
6,9,9
7,10,57
8,11,21
9,12,57


In [5]:
# Перевірка grain і повноти: сума trip_count збігається з кількістю рядків джерела
show('''
SELECT
    (SELECT COUNT(*) FROM nyc_taxi_raw) AS raw_rows,
    SUM(trip_count) AS sum_trip_count,
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT pickup_zone_id) AS n_zones,
    MIN(ts_day) AS first_day,
    MAX(ts_day) AS last_day,
    COUNT(DISTINCT ts_day) AS n_days
FROM t7_hw_taxi_daily;
''')

,raw_rows,sum_trip_count,daily_rows,n_zones,first_day,last_day,n_days
0,481862,481862,8487,240,2024-02-01,2024-03-31,60


Сума `trip_count` дорівнює кількості сирих рядків, тобто агрегація нічого не загубила і не розмножила. У таблиці 8 487 рядків (умова вимагає понад 100), 240 зон і 60 днів. Вибірка найпомітніша на рідкісних зонах: зона 1 у виводі вище має лише 10 днів із 60, а зони 4 і 7 усі 60. Зони 264 і 265 у довіднику TLC означають «Unknown» і «Outside of NYC». Вони лишаються, бо SQL з умови їх не фільтрує, але для реальної моделі це не географічні зони.

### Демонстраційна typed-таблиця з TIMESTAMPTZ

In [6]:
run('''
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;
CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;
''')

show('''
SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5;
''')

,pickup_at,pickup_type
0,2024-02-15 16:11:19+00:00,timestamp with time zone
1,2024-02-17 20:02:44+00:00,timestamp with time zone
2,2024-02-18 21:05:09+00:00,timestamp with time zone
3,2024-02-04 05:48:15+00:00,timestamp with time zone
4,2024-02-18 21:40:48+00:00,timestamp with time zone


TLC записує локальний час Нью-Йорка без зони. `AT TIME ZONE 'America/New_York'` перетворює його на момент часу, а сесія показує цей момент у UTC, тому в лютому години зсунуті на 5, а після переходу на літній час 10 березня на 4. Денна агрегація свідомо лишається в локальному часі: добовий попит у зоні формується за нью-йоркською, а не UTC-добою.

## Завдання 2. Time-series ML features

### Feature 1. Lag features

In [7]:
show('''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-02-04,1,NaN,NaN,None
1,1,2024-02-21,1,1.0,NaN,None
2,1,2024-02-25,1,1.0,NaN,None
3,1,2024-03-10,2,1.0,NaN,None
4,1,2024-03-11,1,2.0,NaN,None
5,1,2024-03-14,1,1.0,NaN,None
6,1,2024-03-21,1,1.0,NaN,None
7,1,2024-03-24,1,1.0,1.0,None
8,1,2024-03-25,1,1.0,1.0,None
9,1,2024-03-31,1,1.0,1.0,None


`LAG(x, n)` відступає на n рядків, а не на n днів. Це те саме лише тоді, коли в зоні немає пропущених днів. Перевірка на реальних даних:

In [8]:
show('''
WITH lagged AS (
    SELECT
        pickup_zone_id,
        ts_day,
        LAG(ts_day, 1)  OVER w AS day_lag_1,
        LAG(ts_day, 7)  OVER w AS day_lag_7,
        LAG(ts_day, 30) OVER w AS day_lag_30
    FROM t7_hw_taxi_daily
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
)
SELECT
    COUNT(day_lag_1) AS rows_lag_1,
    COUNT(*) FILTER (WHERE ts_day - day_lag_1 <> 1) AS wrong_lag_1,
    COUNT(day_lag_7) AS rows_lag_7,
    COUNT(*) FILTER (WHERE ts_day - day_lag_7 <> 7) AS wrong_lag_7,
    COUNT(day_lag_30) AS rows_lag_30,
    COUNT(*) FILTER (WHERE ts_day - day_lag_30 <> 30) AS wrong_lag_30,
    (SELECT COUNT(*) FROM (
        SELECT pickup_zone_id FROM t7_hw_taxi_daily
        GROUP BY pickup_zone_id HAVING COUNT(*) < 60
    ) AS z) AS zones_with_gaps
FROM lagged;
''')

,rows_lag_1,wrong_lag_1,rows_lag_7,wrong_lag_7,rows_lag_30,wrong_lag_30,zones_with_gaps
0,8247,1677,6896,2725,2886,860,175


Це тиха помилка того самого типу, що й відсутній `PARTITION BY`: 860 із 2 886 значень «лагу 30 днів» насправді дивляться на іншу дистанцію, бо 175 із 240 зон мають дні без рядка. Тому у фінальному view лаги рахуються після доповнення календарем (див. нижче). Окремі запити Feature 2–4 нижче лишаються на `t7_hw_taxi_daily`, як в умові. Для Feature 2 і 3 межі вікна від цього не залежать: `RANGE ... INTERVAL` рахує календарні дні. У Feature 4 `expanding_avg_trips` тут усереднює лише дні з рядком, а у view всі календарні дні.

### Feature 2. Rolling 7-day average fare

In [9]:
show('''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-02-04,25.0,25.000000,1
1,1,2024-02-21,120.0,120.000000,1
2,1,2024-02-25,25.0,72.500000,2
3,1,2024-03-10,23.0,23.000000,1
4,1,2024-03-11,3.0,13.000000,2
5,1,2024-03-14,87.4,37.800000,3
6,1,2024-03-21,68.1,68.100000,1
7,1,2024-03-24,35.0,51.550000,2
8,1,2024-03-25,8.6,37.233333,3
9,1,2024-03-31,19.0,13.800000,2


`days_in_window` менше 7 на початку періоду і там, де в календарному вікні є дні без рядка. `ROWS BETWEEN 6 PRECEDING` тут дав би вікно ширше за 7 днів. Це середнє денних середніх, а не зважене за поїздками: день з однією поїздкою важить стільки ж, скільки день із сотнею.

### Feature 3. Rolling 30-day fare volatility

In [10]:
show('''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-02-04,25.0,NaN,1
1,1,2024-02-21,120.0,67.175144,2
2,1,2024-02-25,25.0,54.848276,3
3,1,2024-03-10,23.0,55.434646,3
4,1,2024-03-11,3.0,52.449182,4
5,1,2024-03-14,87.4,49.617658,5
6,1,2024-03-21,68.1,44.882799,6
7,1,2024-03-24,35.0,31.445175,6
8,1,2024-03-25,8.6,31.098270,7
9,1,2024-03-31,19.0,31.523045,7


На першому рядку зони `STDDEV_SAMP` дає NULL: вибіркове відхилення не визначене для одного значення. У рідкісних зонах `avg_fare` дня рахується з однієї-двох поїздок (у зони 1 у виводі від 3 до 120), тому таке відхилення показує радше шум вибірки, ніж коливання цін.

### Feature 4. Expanding cumulative trips

In [11]:
show('''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-02-04,1,1,1.000000
1,1,2024-02-21,1,2,1.000000
2,1,2024-02-25,1,3,1.000000
3,1,2024-03-10,2,5,1.250000
4,1,2024-03-11,1,6,1.200000
5,1,2024-03-14,1,7,1.166667
6,1,2024-03-21,1,8,1.142857
7,1,2024-03-24,1,9,1.125000
8,1,2024-03-25,1,10,1.111111
9,1,2024-03-31,1,11,1.100000


### Combined feature view

Структура, вікна і назви колонок взяті з умови, але перед обчисленням ознак додано CTE `daily_dense`: кожна зона зводиться до повного календаря від першого до останнього дня таблиці (межі з даних, без `CURRENT_DATE`). Для дня без рядка `trip_count = 0` і `total_distance = 0`. Це нуль у вибірці: у рідкісної зони реальні поїздки того дня могли бути. `avg_fare` лишається NULL: ціни без поїздок немає, і `AVG` та `STDDEV_SAMP` такі дні пропускають. Тепер один рядок дорівнює одному дню, тому `LAG(..., 30)` означає рівно 30 днів. `SELECT *` замінено явним переліком колонок, щоб склад view було видно в коді, а `CREATE OR REPLACE VIEW` на `DROP VIEW IF EXISTS` і `CREATE VIEW`, бо `OR REPLACE` не дозволяє змінювати наявні колонки.

In [12]:
run('''
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
CREATE VIEW v_t7_hw_taxi_features AS
WITH calendar AS (
    SELECT generate_series(
        MIN(ts_day)::TIMESTAMP,
        MAX(ts_day)::TIMESTAMP,
        INTERVAL '1 day'
    )::DATE AS ts_day
    FROM t7_hw_taxi_daily
),
zones AS (
    SELECT DISTINCT pickup_zone_id
    FROM t7_hw_taxi_daily
),
daily_dense AS (
    SELECT
        z.pickup_zone_id,
        c.ts_day,
        COALESCE(d.trip_count, 0) AS trip_count,
        d.avg_fare,
        COALESCE(d.total_distance, 0) AS total_distance
    FROM zones AS z
    CROSS JOIN calendar AS c
    LEFT JOIN t7_hw_taxi_daily AS d
        ON d.pickup_zone_id = z.pickup_zone_id
       AND d.ts_day = c.ts_day
),
features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,
        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM daily_dense
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    avg_fare,
    total_distance,
    trips_lag_1d,
    trips_lag_7d,
    trips_lag_30d,
    rolling_avg_fare_7d,
    rolling_std_fare_30d,
    cumulative_trips,
    expanding_avg_trips
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
''')

show('''
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
''')

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-02,0,NaN,0.00,0,0,0,25.0,54.848276,3,0.096774
1,1,2024-03-03,0,NaN,0.00,0,1,0,NaN,54.848276,3,0.093750
2,1,2024-03-04,0,NaN,0.00,0,0,0,NaN,54.848276,3,0.090909
3,1,2024-03-05,0,NaN,0.00,0,0,1,NaN,67.175144,3,0.088235
4,1,2024-03-06,0,NaN,0.00,0,0,0,NaN,67.175144,3,0.085714
5,1,2024-03-07,0,NaN,0.00,0,0,0,NaN,67.175144,3,0.083333
6,1,2024-03-08,0,NaN,0.00,0,0,0,NaN,67.175144,3,0.081081
7,1,2024-03-09,0,NaN,0.00,0,0,0,NaN,67.175144,3,0.078947
8,1,2024-03-10,2,23.0,3.32,0,0,0,23.0,55.434646,5,0.128205
9,1,2024-03-11,1,3.0,17.40,2,0,0,13.0,52.449182,6,0.150000


In [13]:
# Усі вікна view: кожне визначення містить PARTITION BY pickup_zone_id
view_sql = show("SELECT pg_get_viewdef('v_t7_hw_taxi_features'::regclass, true) AS d")['d'][0]
windows = re.findall(r'(\w+) AS \(PARTITION BY (\S*pickup_zone_id)', re.sub(r'\s+', ' ', view_sql))
used = re.findall(r'\bOVER\s+(\S+)', view_sql)
print('window definitions with PARTITION BY pickup_zone_id:', [w[0] for w in windows])
print('window function calls:', len(used), '| windows they use:', sorted(set(used)))
assert set(used) == {w[0] for w in windows}

show('''
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT pickup_zone_id) AS n_zones,
    MIN(ts_day) AS first_day,
    MAX(ts_day) AS last_day,
    COUNT(*) FILTER (WHERE trip_count = 0) AS zero_trip_rows,
    COUNT(*) - COUNT(rolling_avg_fare_7d) AS null_rolling_avg_7d,
    COUNT(*) - COUNT(rolling_std_fare_30d) AS null_rolling_std_30d
FROM v_t7_hw_taxi_features;
''')

window definitions with PARTITION BY pickup_zone_id: ['w_pid', 'w7', 'w30', 'we']
window function calls: 7 | windows they use: ['w30', 'w7', 'w_pid', 'we']


,n_rows,n_zones,first_day,last_day,zero_trip_rows,null_rolling_avg_7d,null_rolling_std_30d
0,7200,240,2024-03-02,2024-03-31,2696,621,405


View містить 240 зон × 30 днів = 7 200 рядків: перші 30 із 60 днів відсічені фільтром `trips_lag_30d IS NOT NULL`, тож навчальний набір починається 2 березня. NULL у rolling-ознаках припадає на рідкісні зони, де за вікно у вибірку потрапило надто мало поїздок. Модель має або обробляти їх, або фільтрувати такі зони.

Незалежна перевірка: ті самі ознаки рахуються в pandas з `t7_hw_taxi_daily` і порівнюються з view по всіх рядках.

In [14]:
daily = show('SELECT pickup_zone_id, ts_day, trip_count, avg_fare FROM t7_hw_taxi_daily')
daily['ts_day'] = pd.to_datetime(daily['ts_day'])
daily['avg_fare'] = daily['avg_fare'].astype(float)

full_index = pd.MultiIndex.from_product(
    [daily['pickup_zone_id'].unique(), pd.date_range(daily['ts_day'].min(), daily['ts_day'].max())],
    names=['pickup_zone_id', 'ts_day'],
)
check = daily.set_index(['pickup_zone_id', 'ts_day']).reindex(full_index).sort_index().reset_index()
check['trip_count'] = check['trip_count'].fillna(0)

by_zone = check.groupby('pickup_zone_id')
check['trips_lag_1d'] = by_zone['trip_count'].shift(1)
check['trips_lag_7d'] = by_zone['trip_count'].shift(7)
check['trips_lag_30d'] = by_zone['trip_count'].shift(30)
check['rolling_avg_fare_7d'] = by_zone['avg_fare'].transform(lambda s: s.rolling(7, min_periods=1).mean())
check['rolling_std_fare_30d'] = by_zone['avg_fare'].transform(lambda s: s.rolling(30, min_periods=2).std())
check['cumulative_trips'] = by_zone['trip_count'].cumsum()
check['expanding_avg_trips'] = by_zone['trip_count'].transform(lambda s: s.expanding().mean())

view = show('SELECT * FROM v_t7_hw_taxi_features')
view['ts_day'] = pd.to_datetime(view['ts_day'])
merged = view.merge(check, on=['pickup_zone_id', 'ts_day'], suffixes=('_sql', '_pd'), validate='one_to_one')

feature_columns = ['trips_lag_1d', 'trips_lag_7d', 'trips_lag_30d', 'rolling_avg_fare_7d',
                   'rolling_std_fare_30d', 'cumulative_trips', 'expanding_avg_trips']
mismatches = {
    name: int((~np.isclose(merged[f'{name}_sql'].astype(float), merged[f'{name}_pd'].astype(float),
                           equal_nan=True, atol=1e-6)).sum())
    for name in feature_columns
}
print('rows compared:', len(merged))
print('mismatches per feature:', mismatches)
assert len(merged) == len(view) and not any(mismatches.values())

rows compared: 7200
mismatches per feature: {'trips_lag_1d': 0, 'trips_lag_7d': 0, 'trips_lag_30d': 0, 'rolling_avg_fare_7d': 0, 'rolling_std_fare_30d': 0, 'cumulative_trips': 0, 'expanding_avg_trips': 0}


### Leakage у rolling- і expanding-ознаках

Frame `... AND CURRENT ROW` з умови включає поточний день. Якщо ціль моделі стосується цього ж дня (наприклад, `trip_count` дня D), то такі ознаки містять інформацію з майбутнього: `avg_fare` і `cumulative_trips` дня D на момент прогнозу ще невідомі. Лаги цієї проблеми не мають. Для такої цілі ознаки треба зсунути на день:

In [15]:
show('''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    rolling_avg_fare_7d,
    LAG(rolling_avg_fare_7d, 1) OVER w AS rolling_avg_fare_7d_prev_day,
    cumulative_trips,
    LAG(cumulative_trips, 1) OVER w AS cumulative_trips_prev_day
FROM v_t7_hw_taxi_features
WHERE pickup_zone_id = 132
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY ts_day
LIMIT 5;
''')

,pickup_zone_id,ts_day,trip_count,rolling_avg_fare_7d,rolling_avg_fare_7d_prev_day,cumulative_trips,cumulative_trips_prev_day
0,132,2024-03-02,303,63.735714,NaN,10006,NaN
1,132,2024-03-03,404,64.071429,63.735714,10410,10006.0
2,132,2024-03-04,413,63.990000,64.071429,10823,10410.0
3,132,2024-03-05,296,63.904286,63.990000,11119,10823.0
4,132,2024-03-06,294,63.957143,63.904286,11413,11119.0


## Завдання 3. JSONB-таблиця для подій

Генератор подій узято з умови без змін, 50 000 рядків.

In [16]:
run('''
DROP TABLE IF EXISTS t7_hw_events;
CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);
INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;
ANALYZE t7_hw_events;
''')

show('''
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
''')

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [17]:
# Кожен тип події має власну схему payload
show('''
WITH key_sets AS (
    SELECT
        event_type,
        (SELECT string_agg(k, ', ' ORDER BY k) FROM jsonb_object_keys(payload) AS k) AS payload_keys
    FROM t7_hw_events
)
SELECT event_type, payload_keys, COUNT(*) AS n_events
FROM key_sets
GROUP BY event_type, payload_keys
ORDER BY event_type;
''')

,event_type,payload_keys,n_events
0,add_to_cart,"price, product_id, quantity, tags",10000
1,checkout,"items_count, order_id, payment_method, total",10000
2,click,"device, element_id, page_url",10000
3,page_view,"device, referrer, url",10000
4,recommendation_shown,"experiment, model_name, model_version, recommendations",10000


### JSONB Query 1. Containment `@>`

Де запит з умови має `LIMIT` без сортування, додано `ORDER BY event_id`, щоб результат був відтворюваним.

In [18]:
show('''
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
ORDER BY event_id
LIMIT 10;
''')

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


### JSONB Query 2. Key existence `?` та `?|`

In [19]:
show('''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
ORDER BY event_id
LIMIT 10;
''')

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [20]:
show('''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
ORDER BY event_id
LIMIT 10;
''')

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


### JSONB Query 3. JSONPath

In [21]:
show('''
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
ORDER BY event_id
LIMIT 15;
''')

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [22]:
show('''
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
ORDER BY event_id
LIMIT 10;
''')

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


## Завдання 4. GIN-index demo + EXPLAIN ANALYZE

Методика. Обсяг лишається 50 000 рядків з умови, бо planner і на ньому сам обирає індекси. Параметри planner не змінюються (`enable_seqscan` тощо не чіпаю). Замість простого `ANALYZE` на старті (і для `t7_hw_checkouts_norm` у Завданні 5) виконується `VACUUM (ANALYZE)`: він фіксує hint bits і visibility map, інакше фоновий autovacuum міг би змінити `Buffers` між замірами до і після. Кожен EXPLAIN виконується 7 разів поспіль, показано план запуску з медіанним часом і діапазон часу всіх запусків.

### Step 1. Containment query без GIN-індексу

In [23]:
run('''
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
DROP INDEX IF EXISTS idx_t7_events_type_btree;
''')
run_autocommit('VACUUM (ANALYZE) t7_hw_events')

explain('containment_before', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
''')

[containment_before]
Aggregate  (cost=1759.03..1759.04 rows=1 width=8) (actual time=101.432..101.433 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3612 width=0) (actual time=0.015..101.056 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning Time: 0.129 ms
Execution Time: 101.462 ms
Median of 7 runs shown; Execution Time range 68.566..144.420 ms


`Seq Scan`: прочитано всі 1000 сторінок, відкинуто 46666 рядків. `jsonb_build_object(...)` має volatility STABLE, тому у `Filter` він обчислюється заново для кожного рядка. Щоб відокремити цю ціну від ціни самої перевірки `@>`, той самий запит із готовою константою:

In [24]:
explain('containment_before_literal', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"payment_method": "card"}'::jsonb;
''')

[containment_before_literal]
Aggregate  (cost=1634.03..1634.04 rows=1 width=8) (actual time=18.704..18.705 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=3612 width=0) (actual time=0.013..18.422 rows=3334 loops=1)
        Filter: (payload @> '{"payment_method": "card"}'::jsonb)
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning Time: 0.119 ms
Execution Time: 18.737 ms
Median of 7 runs shown; Execution Time range 16.203..49.067 ms


З константою час 18.737 ms проти 101.462 ms. Значна частина часу запиту з умови припадає на побудову `jsonb_build_object`, а не на `@>`, тому після індексу нижче порівнюю обидва варіанти.

### Step 2. GIN-індекс для containment / JSONPath workload

In [25]:
run('''
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
ANALYZE t7_hw_events;
''')

show('''
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;
''')

,gin_index_size
0,1144 kB


### Step 3. Той самий containment query після GIN

In [26]:
explain('containment_after', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
''')

[containment_after]
Aggregate  (cost=1084.20..1084.21 rows=1 width=8) (actual time=15.875..15.878 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.47..1076.81 rows=2956 width=0) (actual time=0.509..15.596 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.73 rows=2956 width=0) (actual time=0.363..0.364 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.168 ms
Execution Time: 15.922 ms
Median of 7 runs shown; Execution Time range 6.546..28.130 ms


In [27]:
explain('referrer_after', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
''')

[referrer_after]
Aggregate  (cost=1083.33..1083.34 rows=1 width=8) (actual time=9.296..9.298 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.27..1076.04 rows=2918 width=0) (actual time=0.539..8.925 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.54 rows=2918 width=0) (actual time=0.376..0.376 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.174 ms
Execution Time: 9.361 ms
Median of 7 runs shown; Execution Time range 5.992..10.842 ms


`Seq Scan` замінився на `Bitmap Index Scan` + `Bitmap Heap Scan`: cost 1759.04 → 1084.21, Execution Time 101.462 ms → 15.922 ms. Але `Buffers` не зменшились (1000 → 1004), а `Heap Blocks: exact=1000` означає, що прочитано всю таблицю. Генератор рівномірно розкидає кожну 15-ту подію по всіх сторінках, тож на кожній є хоч один збіг. Виграш тут не в I/O, а в CPU: перевіряється 3334 рядки замість 50 000. Оцінка planner: 2956 рядків проти реальних 3334, але для вибору плану цієї точності вистачило.

GIN для `@>` неточний: `jsonb_path_ops` зберігає хеші шляхів зі значеннями, тому кожен кандидат перевіряється в heap. Це той самий `Recheck Cond`, де для запиту з умови знову виконується `jsonb_build_object`, але вже для 3334 кандидатів. Варіант із константою показує виграш без цієї складової:

In [28]:
explain('containment_after_literal', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"payment_method": "card"}'::jsonb;
''')

[containment_after_literal]
Aggregate  (cost=1076.81..1076.82 rows=1 width=8) (actual time=5.400..5.401 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.47..1069.42 rows=2956 width=0) (actual time=0.871..5.135 rows=3334 loops=1)
        Recheck Cond: (payload @> '{"payment_method": "card"}'::jsonb)
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.73 rows=2956 width=0) (actual time=0.363..0.364 rows=3334 loops=1)
              Index Cond: (payload @> '{"payment_method": "card"}'::jsonb)
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.199 ms
Execution Time: 5.444 ms
Median of 7 runs shown; Execution Time range 4.534..12.556 ms


З константою Execution Time 18.737 ms → 5.444 ms. Це чесніша оцінка ефекту саме індексу, ніж 101.462 ms → 15.922 ms.

Контрольний запит із високою селективністю (рівно одне замовлення) показує, де GIN економить саме I/O:

In [29]:
explain('containment_selective_after', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('order_id', 'o_100003');
''')

[containment_selective_after]
Aggregate  (cost=24.40..24.41 rows=1 width=8) (actual time=0.018..0.018 rows=1 loops=1)
  Buffers: shared hit=4
  ->  Bitmap Heap Scan on t7_hw_events  (cost=12.84..24.39 rows=3 width=0) (actual time=0.015..0.016 rows=1 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('order_id', 'o_100003'))
        Heap Blocks: exact=1
        Buffers: shared hit=4
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..12.84 rows=3 width=0) (actual time=0.009..0.009 rows=1 loops=1)
              Index Cond: (payload @> jsonb_build_object('order_id', 'o_100003'))
              Buffers: shared hit=3
Planning:
  Buffers: shared hit=1
Planning Time: 0.093 ms
Execution Time: 0.037 ms
Median of 7 runs shown; Execution Time range 0.033..0.219 ms


`Buffers` 4 замість 1000, `Heap Blocks: exact=1`.

GIN підтримує не все, що виглядає як JSONPath. Три варіанти одного типу питання за наявного `jsonb_path_ops`:

In [30]:
explain('jsonpath_operator', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @? '$.payment_method ? (@ == "card")';
''')

[jsonpath_operator]
Aggregate  (cost=1076.81..1076.82 rows=1 width=8) (actual time=8.580..8.582 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.47..1069.42 rows=2956 width=0) (actual time=1.490..7.374 rows=3334 loops=1)
        Recheck Cond: (payload @? '$."payment_method"?(@ == "card")'::jsonpath)
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.73 rows=2956 width=0) (actual time=0.355..0.356 rows=3334 loops=1)
              Index Cond: (payload @? '$."payment_method"?(@ == "card")'::jsonpath)
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.141 ms
Execution Time: 8.615 ms
Median of 7 runs shown; Execution Time range 4.973..14.402 ms


In [31]:
explain('jsonpath_function', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE jsonb_path_exists(payload, '$.payment_method ? (@ == "card")');
''')

[jsonpath_function]
Aggregate  (cost=1666.67..1666.68 rows=1 width=8) (actual time=31.840..31.841 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=16667 width=0) (actual time=0.013..31.518 rows=3334 loops=1)
        Filter: jsonb_path_exists(payload, '$."payment_method"?(@ == "card")'::jsonpath, '{}'::jsonb, false)
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning Time: 0.074 ms
Execution Time: 31.875 ms
Median of 7 runs shown; Execution Time range 17.128..44.830 ms


In [32]:
explain('jsonpath_range', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @? '$.total ? (@ > 300)';
''')

[jsonpath_range]
Aggregate  (cost=1637.32..1637.33 rows=1 width=8) (actual time=39.334..39.336 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=4927 width=0) (actual time=0.101..35.897 rows=5000 loops=1)
        Filter: (payload @? '$."total"?(@ > 300)'::jsonpath)
        Rows Removed by Filter: 45000
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=1
Planning Time: 0.161 ms
Execution Time: 39.369 ms
Median of 7 runs shown; Execution Time range 19.334..56.265 ms


Оператор `@?` з рівністю йде через індекс. Функція `jsonb_path_exists` з умови дає `Seq Scan`: індекс прив'язаний до оператора, а не до функції. Діапазон `@ > 300` теж дає `Seq Scan`: GIN-класи JSONB вміють шукати лише рівність значень, для порівнянь потрібен expression B-tree на `(payload ->> 'total')::NUMERIC`.

### Step 4. Коли GIN не допомагає: path-specific text equality

In [33]:
explain('device_before', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
''')

[device_before]
Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=21.668..21.669 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.018..21.058 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.084 ms
Execution Time: 21.700 ms
Median of 7 runs shown; Execution Time range 16.644..43.653 ms


GIN існує, але план `Seq Scan`: оператори `->>` і `=` не входять до opclass `jsonb_path_ops`. Зверніть увагу на оцінку: 250 рядків проти реальних 8333. Статистики на вираз ще немає, тому planner бере типову селективність 0.5%.

In [34]:
run('''
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));
ANALYZE t7_hw_events;
''')

explain('device_after', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
''')

[device_after]
Aggregate  (cost=1240.68..1240.69 rows=1 width=8) (actual time=3.981..3.982 rows=1 loops=1)
  Buffers: shared hit=11
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.25..1220.05 rows=8253 width=0) (actual time=0.500..3.443 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=11
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.19 rows=8253 width=0) (actual time=0.368..0.368 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared hit=9
Planning Time: 0.093 ms
Execution Time: 4.013 ms
Median of 7 runs shown; Execution Time range 2.925..6.084 ms


`Seq Scan` → `Bitmap Index Scan on idx_t7_events_device_btree`: cost 1750.63 → 1240.69, час 21.700 ms → 4.013 ms, `Buffers` 1000 → 11. Оцінка стала точною (8253 проти 8333), бо `ANALYZE` збирає статистику по індексованому виразу.

Падіння `Buffers` пояснюється тим, що B-tree, на відміну від GIN, дає точну відповідь. Рядок `Recheck Cond` друкується в плані і тут, але перевіряти нічого: `COUNT(*)` не потребує колонок, а сторінки all-visible після `VACUUM`, тому heap не читається, хоча `Heap Blocks` усе ще лічить усі сторінки з бітової карти. Для порівняння, те саме питання через GIN (логічно еквівалентне, бо `device` завжди рядок):

In [35]:
explain('device_via_gin', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('device', 'mobile');
''')

[device_via_gin]
Aggregate  (cost=1175.28..1175.29 rows=1 width=8) (actual time=21.971..21.972 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=56.66..1158.33 rows=6778 width=0) (actual time=0.966..21.380 rows=8333 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('device', 'mobile'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..54.97 rows=6778 width=0) (actual time=0.812..0.813 rows=8333 loops=1)
              Index Cond: (payload @> jsonb_build_object('device', 'mobile'))
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=1
Planning Time: 0.157 ms
Execution Time: 22.008 ms
Median of 7 runs shown; Execution Time range 16.807..48.344 ms


Рядків стільки ж (8333), але 1006 buffers і 22.008 ms проти 11 buffers і 4.013 ms у B-tree: тут GIN мусить перевірити кожного кандидата в heap. Форма запиту визначає індекс: `->>` може використати лише expression index, `@>` лише GIN.

### Додатково: `jsonb_path_ops` проти `jsonb_ops`

`idx_t7_events_payload_gin_ops` згадано в `DROP INDEX` умови, тому порівнюю обидва класи на key existence. Умова попереджає, що `?` потребує default `jsonb_ops`. Перевірка:

In [36]:
explain('key_exists_path_ops', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags';
''')

[key_exists_path_ops]
Aggregate  (cost=1650.05..1650.06 rows=1 width=8) (actual time=13.459..13.460 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=10019 width=0) (actual time=0.010..12.712 rows=10000 loops=1)
        Filter: (payload ? 'tags'::text)
        Rows Removed by Filter: 40000
        Buffers: shared hit=1000
Planning Time: 0.118 ms
Execution Time: 13.578 ms
Median of 7 runs shown; Execution Time range 11.697..21.030 ms


In [37]:
run('''
CREATE INDEX idx_t7_events_payload_gin_ops
    ON t7_hw_events USING GIN (payload);
ANALYZE t7_hw_events;
''')

explain('key_exists_jsonb_ops', '''
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags';
''')

[key_exists_jsonb_ops]
Aggregate  (cost=1222.53..1222.54 rows=1 width=8) (actual time=8.455..8.457 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=76.36..1198.17 rows=9745 width=0) (actual time=1.227..7.805 rows=10000 loops=1)
        Recheck Cond: (payload ? 'tags'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_ops  (cost=0.00..73.92 rows=9745 width=0) (actual time=1.069..1.069 rows=10000 loops=1)
              Index Cond: (payload ? 'tags'::text)
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=1
Planning Time: 0.135 ms
Execution Time: 8.494 ms
Median of 7 runs shown; Execution Time range 8.292..16.596 ms


In [38]:
show('''
SELECT
    indexrelname AS index_name,
    pg_size_pretty(pg_relation_size(indexrelid)) AS index_size,
    pg_relation_size(indexrelid) AS index_bytes
FROM pg_stat_user_indexes
WHERE relname = 't7_hw_events'
UNION ALL
SELECT 'table t7_hw_events (heap)',
       pg_size_pretty(pg_relation_size('t7_hw_events')),
       pg_relation_size('t7_hw_events')
ORDER BY index_bytes DESC;
''')

,index_name,index_size,index_bytes
0,table t7_hw_events (heap),8000 kB,8192000
1,idx_t7_events_payload_gin_ops,1696 kB,1736704
2,idx_t7_events_payload_gin_pathops,1144 kB,1171456
3,t7_hw_events_pkey,1112 kB,1138688
4,idx_t7_events_device_btree,360 kB,368640


З `jsonb_path_ops` запит `?` йде через `Seq Scan`, з `jsonb_ops` через `Bitmap Index Scan` (13.578 ms → 8.494 ms). Зате `jsonb_ops` більший: 1696 kB проти 1144 kB.

### Додатково: ціна запису

Читання вимірюють EXPLAIN-и вище. Для запису одна й та сама вставка 50 000 рядків у порожню копію таблиці повторюється 5 разів для кожного набору індексів, у таблиці медіана часу.

In [39]:
run('DROP TABLE IF EXISTS t7_hw_events_write_test')
run('CREATE TABLE t7_hw_events_write_test (LIKE t7_hw_events INCLUDING DEFAULTS)')

setups = {
    'no extra index': None,
    'GIN jsonb_path_ops': 'CREATE INDEX t7_hw_write_gin ON t7_hw_events_write_test USING GIN (payload jsonb_path_ops)',
    'GIN jsonb_ops': 'CREATE INDEX t7_hw_write_gin ON t7_hw_events_write_test USING GIN (payload)',
    'B-tree (payload ->> device)': "CREATE INDEX t7_hw_write_gin ON t7_hw_events_write_test ((payload ->> 'device'))",
}
insert_sql = '''
INSERT INTO t7_hw_events_write_test (event_id, user_id, event_ts, event_type, payload)
SELECT event_id, user_id, event_ts, event_type, payload FROM t7_hw_events
'''

write_rows = []
for name, ddl in setups.items():
    run('DROP INDEX IF EXISTS t7_hw_write_gin')
    if ddl:
        run(ddl)
    timings = []
    for _ in range(5):
        run('TRUNCATE t7_hw_events_write_test')
        started = time.perf_counter()
        run(insert_sql)
        timings.append((time.perf_counter() - started) * 1000)
    write_rows.append({'indexes': name, 'median_insert_ms': round(statistics.median(timings), 1)})

run('DROP TABLE t7_hw_events_write_test')
write_cost = pd.DataFrame(write_rows)
write_cost['vs_no_index'] = (write_cost['median_insert_ms'] / write_cost['median_insert_ms'].iloc[0]).round(2)
write_cost

,indexes,median_insert_ms,vs_no_index
0,no extra index,210.0,1.00
1,GIN jsonb_path_ops,194.9,0.93
2,GIN jsonb_ops,444.7,2.12
3,B-tree (payload ->> device),251.4,1.20


Це вставка однією транзакцією. GIN за замовчуванням має `fastupdate`: частина вставок спершу записується в pending list і потрапляє до індексу пізніше, тому частина ціни переноситься на наступний vacuum або читання.

У цьому запуску `jsonb_path_ops` (194.9 ms) вийшов дешевшим за таблицю без індексу (210.0 ms). Це шум спільної машини Colab, а не економія: індекс не може здешевити запис. Надійно видно лише дорожчий запис у `jsonb_ops` (444.7 ms), а стабільнішим показником ціни лишається розмір індексу.

## Завдання 5. JSONB vs normalized

### Step 1. Нормалізована таблиця для checkout-подій

In [40]:
run('''
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);
INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';
CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);
''')
run_autocommit('VACUUM (ANALYZE) t7_hw_checkouts_norm')

show('''
SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;
''')

,n_checkouts
0,10000


### Step 2. Один запит двома способами

Обидва запити спершу виконуються без EXPLAIN, щоб перевірити, що вони дають один результат.

In [41]:
AGG_JSONB = '''
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
'''

AGG_NORM = '''
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
'''

result_jsonb, result_norm = show(AGG_JSONB), show(AGG_NORM)
print('same result:', result_jsonb.equals(result_norm))
result_jsonb

same result: True


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


In [42]:
explain('agg_jsonb', AGG_JSONB)

[agg_jsonb]
Limit  (cost=1957.78..1957.79 rows=3 width=72) (actual time=10.269..10.272 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1957.78..1971.78 rows=5597 width=72) (actual time=10.268..10.270 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1801.49..1885.44 rows=5597 width=72) (actual time=10.238..10.261 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.21 rows=10085 width=132) (actual time=0.008..6.496 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning Time: 0.108 ms
Execution Time: 10.314 ms
Median of 7 runs shown; Ex

In [43]:
explain('agg_norm', AGG_NORM)

[agg_norm]
Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=2.899..2.901 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=2.898..2.899 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=2.890..2.892 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.008..0.686 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.121 ms
Execution Time: 2.932 ms
Median of 7 runs shown; Execution Time range 2.730..11.022 ms


Прямі цифри не зовсім чесні щодо JSONB: 10.314 ms проти 2.932 ms включають і те, що JSONB-версія сканує всі події (1000 buffers, `Rows Removed by Filter`: 40000), бо на `event_type` немає індексу, а нормалізована таблиця містить лише checkout (94 buffers). Щоб розділити ці ефекти, додаю B-tree на `event_type`:

In [44]:
run('''
CREATE INDEX idx_t7_events_type_btree ON t7_hw_events (event_type);
ANALYZE t7_hw_events;
''')

explain('agg_jsonb_type_index', AGG_JSONB)

[agg_jsonb_type_index]
Limit  (cost=1561.49..1561.50 rows=3 width=72) (actual time=9.339..9.341 rows=3 loops=1)
  Buffers: shared hit=1011
  ->  Sort  (cost=1561.49..1575.16 rows=5469 width=72) (actual time=9.338..9.339 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1011
        ->  HashAggregate  (cost=1408.77..1490.80 rows=5469 width=72) (actual time=9.299..9.328 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1011
              ->  Bitmap Heap Scan on t7_hw_events  (cost=112.76..1260.76 rows=9867 width=132) (actual time=0.480..4.952 rows=10000 loops=1)
                    Recheck Cond: (event_type = 'checkout'::text)
                    Heap Blocks: exact=1000
                    Buffers: shared hit=1011
                    ->  Bitmap Index Scan on idx_t7_events_typ

In [45]:
show('''
SELECT
    't7_hw_events: payload of checkout rows' AS object,
    pg_size_pretty(SUM(pg_column_size(payload))::BIGINT) AS size
FROM t7_hw_events
WHERE event_type = 'checkout'
UNION ALL
SELECT 't7_hw_checkouts_norm: order_id, total, items_count, payment_method',
       pg_size_pretty(SUM(pg_column_size(order_id) + pg_column_size(total)
                          + pg_column_size(items_count) + pg_column_size(payment_method))::BIGINT)
FROM t7_hw_checkouts_norm
UNION ALL
SELECT 't7_hw_checkouts_norm: heap', pg_size_pretty(pg_relation_size('t7_hw_checkouts_norm'));
''')

,object,size
0,t7_hw_events: payload of checkout rows,1038 kB
1,"t7_hw_checkouts_norm: order_id, total, items_count, payment_method",238 kB
2,t7_hw_checkouts_norm: heap,752 kB


Нормалізована версія дала cost 269.07 і 2.932 ms проти cost 1957.79 і 10.314 ms у JSONB. Індекс `idx_t7_checkouts_norm_pm` у плані не бере участі: запит агрегує всю таблицю, тож там `Seq Scan`.

B-tree на `event_type` змінив план на `Bitmap Heap Scan` і знизив cost до 1561.50, час став 9.409 ms. `Buffers` не зменшились (1011): checkout-події є на кожній сторінці, а `payload` усе одно треба читати. Отже, фільтр пояснює лише частину різниці з нормалізованою таблицею, решта припадає на ширину рядка і витягування значень із JSON. Те саме видно в розмірах: payload checkout-рядків 1038 kB проти 238 kB у типізованих колонках. Для `GROUP BY` за виразом немає статистики, тому `HashAggregate` очікує 5597 груп проти 3 реальних, а нормалізована таблиця оцінена точно.

### Step 3. Bonus: MongoDB-equivalent як pseudocode

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL
db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

- PostgreSQL JSONB зручний, коли потрібні SQL, JOIN, транзакції і зв'язок із нормалізованими таблицями: той самий запит можна з'єднати з `t7_hw_checkouts_norm` або таблицею користувачів.
- MongoDB aggregation pipeline природний для document-first workloads: кроки `$match`, `$group`, `$sort` читаються як конвеєр над документами, а схема живе у застосунку.
- High-volume raw clickstream може краще жити в document store або data lake: там дешевше писати великий потік без GIN і без строгої схеми.
- Curated ML features часто зручніше тримати у typed SQL-таблицях: типи, CHECK і `PRIMARY KEY (pickup_zone_id, ts_day)` гарантують контракт ознак для навчання.

## Підсумок вимірів

Таблиця зібрана з планів, збережених вище в цьому запуску. `execution_ms` це Execution Time показаного плану, тобто медіанного запуску.

In [46]:
def plan_summary(label):
    plan = PLANS[label]
    lines = plan.splitlines()
    scan = next(line for line in lines if re.search(r'(Seq Scan|Bitmap Heap Scan|Index Scan|Index Only Scan) on', line))
    index = next((line for line in lines if 'Bitmap Index Scan on' in line), None)
    return {
        'label': label,
        'scan': re.search(r'(Parallel )?(Seq Scan|Bitmap Heap Scan|Index Only Scan|Index Scan)', scan).group(0),
        'index': re.search(r'Bitmap Index Scan on (\S+)', index).group(1) if index else '-',
        'total_cost': float(re.search(r'cost=[\d.]+\.\.([\d.]+)', lines[0]).group(1)),
        'est_rows': int(re.search(r'rows=(\d+) width', scan).group(1)),
        'actual_rows': int(re.search(r'actual time=\S+ rows=(\d+)', scan).group(1)),
        'shared_hit': int(re.search(r'shared hit=(\d+)', plan).group(1)),
        'execution_ms': float(re.search(r'Execution Time: ([\d.]+)', plan).group(1)),
    }


pd.DataFrame([plan_summary(label) for label in PLANS])

,label,scan,index,total_cost,est_rows,actual_rows,shared_hit,execution_ms
0,containment_before,Seq Scan,-,1759.04,3612,3334,1000,101.462
1,containment_before_literal,Seq Scan,-,1634.04,3612,3334,1000,18.737
2,containment_after,Bitmap Heap Scan,idx_t7_events_payload_gin_pathops,1084.21,2956,3334,1004,15.922
3,referrer_after,Bitmap Heap Scan,idx_t7_events_payload_gin_pathops,1083.34,2918,2500,1004,9.361
4,containment_after_literal,Bitmap Heap Scan,idx_t7_events_payload_gin_pathops,1076.82,2956,3334,1004,5.444
5,containment_selective_after,Bitmap Heap Scan,idx_t7_events_payload_gin_pathops,24.41,3,1,4,0.037
6,jsonpath_operator,Bitmap Heap Scan,idx_t7_events_payload_gin_pathops,1076.82,2956,3334,1004,8.615
7,jsonpath_function,Seq Scan,-,1666.68,16667,3334,1000,31.875
8,jsonpath_range,Seq Scan,-,1637.33,4927,5000,1000,39.369
9,device_before,Seq Scan,-,1750.63,250,8333,1000,21.700


## Reflection

**Що змінилось після GIN.** Без індексу запит ішов через `Seq Scan`: total cost 1759.04, Execution Time 101.462 ms, `Rows Removed by Filter` 46666. Після GIN план став `Bitmap Index Scan` + `Bitmap Heap Scan`: cost 1084.21, 15.922 ms. Buffers не впали (1000 → 1004), бо `Heap Blocks: exact=1000`: збіги є на кожній сторінці, тож економиться CPU, а не I/O. Без `jsonb_build_object` у запиті абсолютний виграш менший: 18.737 → 5.444 ms.

**Чому `->>` потребує іншого індексу.** GIN шукає пари шлях-значення для `@>`, а `->>` не входить до його opclass, тому `device_before` лишився `Seq Scan` попри GIN: 21.700 ms, оцінка 250 рядків проти 8333. Expression B-tree дав 4.013 ms і оцінку 8253. Той самий фільтр через `@>` читає heap заради recheck: 1006 buffers проти 11.

**Read/write trade-off.** Швидше читання оплачується записом і місцем. Вставка 50 000 рядків: 210.0 ms без індексу, 444.7 ms із `jsonb_ops`, 251.4 ms із B-tree. `jsonb_path_ops` дав 194.9 ms: це шум Colab, а не економія. Розмір GIN: 1144 kB і 1696 kB за heap 8000 kB. Частину витрат GIN відкладає pending list (`fastupdate`), тож вони повертаються під час vacuum або читання.

**JSONB чи нормалізована таблиця.** JSONB доречний для сирих подій зі змінною схемою. Стабільні поля, з яких будуються ознаки, краще тримати в типізованій таблиці: 2.932 ms і 94 buffers проти 10.314 ms і 1000 buffers, плюс CHECK під час запису. Payload checkout-рядків займає 1038 kB проти 238 kB у колонках. Індекс на `event_type` скоротив JSONB-версію лише до 9.409 ms.

**Новий тип `refund`.** У JSONB достатньо `INSERT` з новим `event_type`: DDL не потрібен, GIN покриває нові ключі, а часті фільтри отримають expression index; перевірити треба споживачів, які не чекають цього типу. У нормалізованому підході потрібні `t7_hw_refunds_norm` із типами, CHECK і `event_id`, індекси, backfill із `t7_hw_events` та оновлення ETL.